# 02 · Optional step rephrasing with Gemma 3 4B (owner: Atharva Hemade)

Section 8.1 and 12: Gemma 3 may rephrase a verified step more simply, **using only the current record**.
A rephrasing is shown only if the mmBERT verifier says it entails the verified text **in both directions**.
This notebook generates rephrasings for every step, runs the verifier, and reports
the share rejected and a 50-per-language sample for the manual critical-error review (Section 17.2).

In [ ]:
!pip -q install transformers accelerate bitsandbytes

In [ ]:
# Clone the repo (or upload it) and move into it.
!git clone https://github.com/<owner>/firstaid-assistant.git 2>/dev/null || true
%cd firstaid-assistant/backend
!python scripts/generate_queries.py

In [ ]:
import json, glob
steps = []
for f in glob.glob("content/situations/*.json"):
    rec = json.load(open(f, encoding="utf-8"))
    for bkey, b in rec["branches"].items():
        for s in b["steps"]:
            for lang, text in s["text"].items():
                if text: steps.append({"situation": rec["id"], "branch": bkey, "id": s["id"], "lang": lang, "text": text})
print(len(steps), "step texts")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
name = "google/gemma-3-4b-it"
tok = AutoTokenizer.from_pretrained(name)
lm = AutoModelForCausalLM.from_pretrained(name, quantization_config=BitsAndBytesConfig(load_in_4bit=True), device_map="auto")
LANG = {"en": "English", "hi": "Hindi", "hinglish": "Hinglish (romanised Hindi)", "mr": "Marathi"}
PROMPT = ("Rewrite this first aid instruction in simpler {lang} words for a panicked bystander. "
          "Keep every number, every action and every warning. Do not add anything. Reply with the rewritten sentence only.\n\nInstruction: {step}")
def rephrase(step, lang):
    msgs = [{"role": "user", "content": PROMPT.format(lang=LANG[lang], step=step)}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(lm.device)
    out = lm.generate(ids, max_new_tokens=96, do_sample=False)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
for s in steps: s["candidate"] = rephrase(s["text"], s["lang"])

In [ ]:
# Verify with the fine-tuned mmBERT from notebook 03 (both directions must entail).
from transformers import AutoTokenizer as T2, AutoModelForSequenceClassification as M2
vt = T2.from_pretrained("../training/out/mmbert_verifier"); vm = M2.from_pretrained("../training/out/mmbert_verifier").eval()
def entails(a, b):
    with torch.no_grad():
        return int(vm(**vt(a, b, return_tensors="pt", truncation=True)).logits.argmax()) == 0
for s in steps: s["accepted"] = entails(s["text"], s["candidate"]) and entails(s["candidate"], s["text"])
import collections
for lang in LANG:
    rows = [s for s in steps if s["lang"] == lang]
    if rows: print(lang, "rejected share =", round(1 - sum(s["accepted"] for s in rows) / len(rows), 3))

In [ ]:
import random
random.seed(0)
sample = [s for lang in LANG for s in random.sample([x for x in steps if x["lang"] == lang], min(50, len([x for x in steps if x["lang"] == lang])))]
json.dump(sample, open("../training/out/rephrase_review_sample.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("Review each pair by hand and record critical errors (changed number, dropped warning, added advice).")